# Stage-1 plate detector — Nepal-specific fine-tune

Trains the ANPR pipeline's **plate locator** on Nepali road imagery.

## Why this exists

`MODEL_REPORT.md` already names the weakness:

> *Plate detector (stage 1) is generic (not Nepal-specific); distant/tiny plates (<40px wide) are skipped by design.*

Stage 2 (the character reader) is trained on 5,298 Nepali images and is strong — mAP50 0.84.
Stage 1 was never trained on Nepali data at all: it is a stock `License_Plate` detector.

Measured on real Kathmandu traffic footage, that stock detector scored **~0.28 confidence on
genuine plates** while scoring **0.75 on burnt-in graphics** in the same frame. Plates are being
missed at stage 1, before the good reader ever sees them. That is the bottleneck this fixes.

## Dataset

`plate_detect_nepali.zip` — 6,927 images, single class `License_Plate`, built by
`build_plate_dataset.py` from two sources:

| Source | Images | What it contributes |
|---|---|---|
| Number plate detection nepali.v1i | 318 | Real road scenes — distant, angled, cluttered |
| 8th sem Proj.v2i (char boxes unioned) | 6,609 | Close-up plates — every layout and colour scheme |

The road-scene set shipped four classes (`'4'`, `'number plate'`, `'numberplate'`, `'objects'`),
but the label histogram showed 93% in one of them and the rest as annotation noise. Stage 1 only
has to answer *"is there a plate here"*, so every box was collapsed to one class.

## Runtime

**Set Runtime → Change runtime type → T4 GPU before running.** On a T4 this is roughly
90 seconds an epoch, so 60 epochs is about 90 minutes.

## 1 · Check the GPU

If this prints nothing, the runtime is CPU-only and training will take many hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Install

Colab ships torch already; only ultralytics is missing.

In [ ]:
!pip install -q ultralytics

import torch, ultralytics
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 3 · Get the data in

Upload **`plate_detect_nepali.zip`** (269 MB) and **`plate_detector.pt`** to your Drive first —
the browser upload widget is slow and dies on a flaky connection, whereas Drive survives a
runtime restart.

Put both at the top level of *My Drive*, then run this.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil, time

DRIVE = '/content/drive/MyDrive'
ZIP   = f'{DRIVE}/plate_detect_nepali.zip'
BASE  = f'{DRIVE}/plate_detector.pt'
DATA  = '/content/plate_detect_nepali'

for path in (ZIP, BASE):
    assert os.path.exists(path), f'Missing {path} — upload it to My Drive first'

# Unzip to local disk, not Drive: Drive is a network mount and reading 6,927
# small files through it every epoch is far slower than the training itself.
if os.path.exists(DATA):
    shutil.rmtree(DATA)

t0 = time.time()
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DATA)
print(f'extracted in {time.time() - t0:.0f}s')

for split in ('train', 'valid', 'test'):
    n = len(os.listdir(f'{DATA}/{split}/images'))
    print(f'{split:6s}: {n} images')

## 4 · Point data.yaml at the Colab path

The yaml was written on Windows and still has the local `path:` in it.

In [ ]:
yaml = f'''# Single-class Nepali plate-LOCATION set for ANPR stage 1.
path: {DATA}
train: train/images
val: valid/images
test: test/images

nc: 1
names: ['License_Plate']
'''

with open(f'{DATA}/data.yaml', 'w') as f:
    f.write(yaml)

print(yaml)

## 5 · Baseline: what the stock detector scores

Run this *before* training. Without a before-number the after-number means nothing, and
"we improved it" is the first claim an examiner will ask you to substantiate.

In [ ]:
from ultralytics import YOLO

baseline = YOLO(BASE).val(data=f'{DATA}/data.yaml', split='test', imgsz=960, device=0)

print('\n=== BASELINE (stock plate_detector.pt) ===')
print(f'mAP50    : {baseline.box.map50:.4f}')
print(f'mAP50-95 : {baseline.box.map:.4f}')
print(f'precision: {baseline.box.mp:.4f}')
print(f'recall   : {baseline.box.mr:.4f}')

## 6 · Train

Starting from the existing detector rather than from scratch: it already knows what a
rectangular plate looks like, and 6,927 images is enough to specialise it but not to teach
detection from nothing.

The augmentation is aimed at the measured failure — small, distant, blurred plates in dense
traffic:

| Setting | Reason |
|---|---|
| `scale=0.6`, `mosaic=1.0` | The failure mode is *distant* plates. Forces many sizes per batch. |
| `degrees=7`, `shear=3` | Nepali plates are frequently mounted crooked. |
| `hsv_s`, `hsv_v` | Red, white, yellow and black plate schemes all occur. |
| `fliplr=0`, `flipud=0` | **Off deliberately.** Plate text is directional — a mirrored plate is not a plate, and training on one teaches a shape that never occurs. |

`patience=15` stops early if it plateaus, so a full 60 epochs is an upper bound rather than
a promise.

In [ ]:
model = YOLO(BASE)

results = model.train(
    data=f'{DATA}/data.yaml',
    epochs=60,
    imgsz=960,
    batch=16,          # a T4 handles 16 at 960px; drop to 8 on an OOM
    device=0,
    workers=2,
    project='/content/runs',
    name='plate_nepali',
    exist_ok=True,

    patience=15,
    optimizer='auto',
    cos_lr=True,

    scale=0.6,
    mosaic=1.0,
    close_mosaic=10,   # last 10 epochs without mosaic, so it finishes on real layouts
    degrees=7.0,
    shear=3.0,
    translate=0.12,
    perspective=0.0005,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.45,
    fliplr=0.0, flipud=0.0,

    plots=True,
    val=True,
)

print('done')

## 7 · Measure the improvement

Same held-out test split, same settings as the baseline — otherwise the comparison is
meaningless.

In [ ]:
BEST = '/content/runs/plate_nepali/weights/best.pt'

tuned = YOLO(BEST).val(data=f'{DATA}/data.yaml', split='test', imgsz=960, device=0)

rows = [
    ('mAP50',     baseline.box.map50, tuned.box.map50),
    ('mAP50-95',  baseline.box.map,   tuned.box.map),
    ('precision', baseline.box.mp,    tuned.box.mp),
    ('recall',    baseline.box.mr,    tuned.box.mr),
]

print(f'{"metric":<10} {"stock":>8} {"tuned":>8} {"change":>9}')
print('-' * 39)
for name, before, after in rows:
    print(f'{name:<10} {before:8.4f} {after:8.4f} {after - before:+9.4f}')

print('\nPaste this table straight into MODEL_REPORT.md section 4.')

## 8 · Eyeball it on real traffic

Numbers on a test split can hide a model that fails on the thing you actually care about.
Upload a frame of dense traffic to `/content/` and compare the two side by side.

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

# Uses test images if you have not uploaded anything of your own.
samples = sorted(glob.glob(f'{DATA}/test/images/road_*'))[:3]
samples = samples or sorted(glob.glob(f'{DATA}/test/images/*'))[:3]

stock, best = YOLO(BASE), YOLO(BEST)

fig, axes = plt.subplots(len(samples), 2, figsize=(15, 5 * len(samples)))
axes = axes.reshape(len(samples), 2)

for row, path in enumerate(samples):
    for col, (label, m) in enumerate([('stock', stock), ('tuned', best)]):
        r = m.predict(path, conf=0.25, imgsz=960, device=0, verbose=False)[0]
        axes[row][col].imshow(Image.fromarray(r.plot()[:, :, ::-1]))
        axes[row][col].set_title(f'{label} — {len(r.boxes)} plates')
        axes[row][col].axis('off')

plt.tight_layout()
plt.show()

## 9 · Save the weights back to Drive

**Do this before the runtime disconnects** — `/content/` is wiped and an hour of training
goes with it.

In [ ]:
OUT = f'{DRIVE}/plate_detector_nepali.pt'
shutil.copy(BEST, OUT)

# The plots are what you want in the report: PR curve, confusion matrix,
# and the loss curves that show it converged rather than stopped early.
shutil.copytree('/content/runs/plate_nepali',
                f'{DRIVE}/plate_nepali_run', dirs_exist_ok=True)

print(f'weights -> {OUT}  ({os.path.getsize(OUT) / 1e6:.1f} MB)')
print(f'run dir -> {DRIVE}/plate_nepali_run')

## 10 · Install it locally

Download `plate_detector_nepali.pt` from Drive and drop it next to the other weights:

```
C:\Users\sauga\Downloads\8th sem Proj.v2i.yolov11\weights\plate_detector_nepali.pt
```

`server.py` already looks for that filename and uses it in preference to the stock detector,
falling back if it is absent — so there is nothing to edit. Restart the ANPR service and
confirm it picked the new weights:

```bash
curl http://127.0.0.1:8000/health
# "plateWeights": "plate_detector_nepali.pt"
# "nepaliPlateDetector": true
```

### If the numbers disappoint

The most likely cause is the road-scene set being small (318 images) next to the crop set
(6,609). If recall on distant plates is still weak, the fix is more *road scenes*, not more
epochs — annotate a few hundred frames of Kathmandu traffic and rebuild. Everything else in
this notebook stays the same.